# 💪 RobustScaler — Quick Notes
---
> **Dataset:** `tips` (seaborn) | **Library:** `sklearn.preprocessing`

## 📌 Key Points
- Scales using **median and IQR** (not mean/std)
- **Robust to outliers** — outliers don't dominate the scaling
- Formula: `x_scaled = (x - median) / IQR`
- IQR = Q3 - Q1 (75th percentile - 25th percentile)
- Best when data has **significant outliers**
- Output range is NOT fixed like MinMax

## 🔑 Scaler Comparison
| Scaler | Formula | Sensitive to Outliers |
|---|---|---|
| StandardScaler | (x-mean)/std | Moderate |
| MinMaxScaler | (x-min)/(max-min) | Very High ❌ |
| **RobustScaler** | **(x-median)/IQR** | **Low ✅** |

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split

# 1. Load & Encode
df = sns.load_dataset('tips')
df = pd.get_dummies(df, columns=['sex','smoker','day','time'], drop_first=True)
df = df.astype(int)

# 2. Inject an outlier to demonstrate
df_outlier = df.copy()
df_outlier.loc[0, 'tip'] = 1000   # artificial outlier

x = df_outlier.drop(columns=['total_bill'])
y = df_outlier['total_bill']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
print("Tip column with outlier:")
print(x_train['tip'].describe().round(2))

In [ ]:
# 3. Compare all 3 scalers on same data
ss  = StandardScaler()
mn  = MinMaxScaler()
rb  = RobustScaler()

tip_col = x_train[['tip']]
tip_ss = ss.fit_transform(tip_col)
tip_mn = mn.fit_transform(tip_col)
tip_rb = rb.fit_transform(tip_col)

print(f"StandardScaler — min: {tip_ss.min():.2f}, max: {tip_ss.max():.2f}")
print(f"MinMaxScaler   — min: {tip_mn.min():.2f}, max: {tip_mn.max():.2f}")
print(f"RobustScaler   — min: {tip_rb.min():.2f}, max: {tip_rb.max():.2f}")

In [ ]:
# 4. Visual comparison
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
data = [tip_col.values, tip_ss, tip_mn, tip_rb]
titles = ['Original (with outlier)', 'StandardScaler', 'MinMaxScaler', 'RobustScaler ✅']
colors = ['steelblue', 'green', 'orange', 'red']

for ax, d, t, c in zip(axes, data, titles, colors):
    ax.hist(d, bins=20, color=c, alpha=0.8)
    ax.set_title(t, fontsize=10)
plt.suptitle('Effect of Outlier on Different Scalers', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# 5. RobustScaler on full dataset (no outlier)
df_clean = sns.load_dataset('tips')
df_clean = pd.get_dummies(df_clean, columns=['sex','smoker','day','time'], drop_first=True)
df_clean = df_clean.astype(int)

x = df_clean.drop(columns=['total_bill'])
y = df_clean['total_bill']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

rb = RobustScaler()
x_train_scaled = rb.fit_transform(x_train)
x_test_scaled  = rb.transform(x_test)

df_rb = pd.DataFrame(x_train_scaled, columns=x_train.columns)
print("RobustScaler output stats:")
print(np.round(df_rb.describe(), 2))

## 🥇 Remember
- `rb.center_` → median of each feature
- `rb.scale_` → IQR of each feature
- Output is NOT bounded to 0-1 or -1 to 1
- Use when you have **outliers you can't remove**
- Common in: financial data, medical data, sensor data
- Still: **fit only on train**, transform both